In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
print("Libraries Imported")

Libraries Imported


In [12]:
df = pd.read_csv('../data/raw/telco_churn.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
pd.set_option('display.max_rows',7043)
pd.set_option('display.max_columns',21)
print("\nFirst look at data:")
df.head(10)

Dataset loaded: 7043 rows, 21 columns

First look at data:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [25]:
print("Handling Missing Values")
print("-"*50)
print("MIssing Values")
print(df.isnull().sum())

#totalcharges is sometimes stored as object we convert it to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'],errors='coerce')

#replacing the missing value with median
median_charge = df['TotalCharges'].median()
df['TotalCharges'].fillna(median_charge,inplace = True)

print("\nMIssing Value Handled!")
print(f"Filled {df['TotalCharges'].isnull().sum()} missing values with median: ${median_charge:.2f}")

Handling Missing Values
--------------------------------------------------
MIssing Values
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

MIssing Value Handled!
Filled 11 missing values with median: $1397.47


In [28]:
print("Dropping Unnecesaary Columns")
print("-"*50)

df_clean = df.drop('customerID',axis=1)
print("Dropped customerID")
print(f"New shape: {df_clean.shape}")

Dropping Unnecesaary Columns
--------------------------------------------------
Dropped customerID
New shape: (7043, 20)


In [30]:
print("STEP 3: ENCODING CATEGORICAL VARIABLES")
print("-"*50)

#copy for encoding
df_encoded = df_clean.copy()

# binary encoding for Yes/No columns
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 
               'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
               'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in binary_cols:
    if col in df_encoded.columns:
        df_encoded[col] = df_encoded[col].map({'Yes': 1, 'No': 0})
        print(f"{col}: Yes=1, No=0")

# Gender
df_encoded['gender'] = df_encoded['gender'].map({'Male': 0, 'Female': 1})
print(f"Gender: Male=0, Female=1")

# MultipleLines
if 'MultipleLines' in df_encoded.columns:
    df_encoded['MultipleLines'] = df_encoded['MultipleLines'].map({
        'No phone service': 0,
        'No': 1,
        'Yes': 2
    })
    print(f"MultipleLines: No phone service=0, No=1, Yes=2")

#InternetService
df_encoded['InternetService'] = df_encoded['InternetService'].map({
    'No': 0,
    'DSL': 1,
    'Fiber optic': 2
})
print(f"InternetService: No=0, DSL=1, Fiber optic=2")

#Contract
df_encoded['Contract'] = df_encoded['Contract'].map({
    'Month-to-month': 0,
    'One year': 1,
    'Two year': 2
})
print(f"Contract: Month-to-month=0, One year=1, Two year=2")

#PaymentMethod
df_encoded['PaymentMethod'] = df_encoded['PaymentMethod'].map({
    'Electronic check': 0,
    'Mailed check': 1,
    'Bank transfer (automatic)': 2,
    'Credit card (automatic)': 3
})
print(f"PaymentMethod: Electronic check=0, Mailed check=1, Bank transfer=2, Credit card=3")

#Target Variable (Churn)
df_encoded['Churn'] = df_encoded['Churn'].map({'No': 0, 'Yes': 1})
print(f"Churn: No=0, Yes=1")

print("\nAll Encoding Done!")

STEP 3: ENCODING CATEGORICAL VARIABLES
--------------------------------------------------
Partner: Yes=1, No=0
Dependents: Yes=1, No=0
PhoneService: Yes=1, No=0
PaperlessBilling: Yes=1, No=0
OnlineSecurity: Yes=1, No=0
OnlineBackup: Yes=1, No=0
DeviceProtection: Yes=1, No=0
TechSupport: Yes=1, No=0
StreamingTV: Yes=1, No=0
StreamingMovies: Yes=1, No=0
Gender: Male=0, Female=1
MultipleLines: No phone service=0, No=1, Yes=2
InternetService: No=0, DSL=1, Fiber optic=2
Contract: Month-to-month=0, One year=1, Two year=2
PaymentMethod: Electronic check=0, Mailed check=1, Bank transfer=2, Credit card=3
Churn: No=0, Yes=1

All Encoding Done!


In [33]:
print("\nChecking for any remaining issues...")
print(f"Missing values: {df_encoded.isnull().sum().sum()}")
print(f"Data types:")
print(df_encoded.dtypes.value_counts())

# Check if all columns are numeric
print(f"\nAll columns numeric: {df_encoded.dtypes.apply(lambda x: x in ['int64', 'float64']).all()}")


Checking for any remaining issues...
Missing values: 9167
Data types:
int64      12
float64     8
Name: count, dtype: int64

All columns numeric: True


In [34]:
print("Splitting Features And Target")
print("-"*50)

# Separate features (X) and target (y)
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nChurn rate: {y.mean()*100:.2f}%")

Splitting Features And Target
--------------------------------------------------
Features (X): (7043, 19)
Target (y): (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Churn rate: 26.54%


In [35]:
print("STEP 5: TRAIN-TEST SPLIT")
print("-"*50)

from sklearn.model_selection import train_test_split

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Keep same churn ratio in both sets
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"\nTraining set churn rate: {y_train.mean()*100:.2f}%")
print(f"Testing set churn rate: {y_test.mean()*100:.2f}%")

print("\nData split successfully!")

STEP 5: TRAIN-TEST SPLIT
--------------------------------------------------
Training set: 5634 samples
Testing set: 1409 samples

Training set churn rate: 26.54%
Testing set churn rate: 26.54%

Data split successfully!


In [37]:
print("STEP 6: HANDLING CLASS IMBALANCE (SMOTE)")
print("-"*70)

from imblearn.over_sampling import SMOTE

# Final check before SMOTE
print("Final check before SMOTE:")
print(f"X_train missing values: {X_train.isnull().sum().sum()}")
print(f"y_train missing values: {y_train.isnull().sum()}")

# If there are ANY NaN values, handle them
if X_train.isnull().sum().sum() > 0 or y_train.isnull().sum() > 0:
    print("\nFound missing values! Handling now...")
    X_train = X_train.fillna(X_train.median())
    X_test = X_test.fillna(X_test.median())
    y_train = y_train.fillna(0)  # Fill target if needed
    y_test = y_test.fillna(0)
    print("Missing values handled!")

print("\nBefore SMOTE:")
print(y_train.value_counts())

# Apply SMOTE
try:
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    
    print("\nAfter SMOTE:")
    print(pd.Series(y_train_balanced).value_counts())
    print(f"\nClasses balanced successfully!")
    
except Exception as e:
    print(f"\n❌ SMOTE Error: {e}")
    print("\nUsing original unbalanced data instead...")
    X_train_balanced = X_train
    y_train_balanced = y_train

STEP 6: HANDLING CLASS IMBALANCE (SMOTE)
----------------------------------------------------------------------
Final check before SMOTE:
X_train missing values: 0
y_train missing values: 0

Before SMOTE:
Churn
0    4139
1    1495
Name: count, dtype: int64

After SMOTE:
Churn
0    4139
1    4139
Name: count, dtype: int64

Classes balanced successfully!


In [39]:
print("Saving Processed Data")
print("-"*70)

df_encoded.to_csv('../data/processed/telco_churn_cleaned.csv', index=False)
print("Saved: data/processed/telco_churn_cleaned.csv")

# Create models folder if it doesn't exist
import os
os.makedirs('../models', exist_ok=True)

# Save the split data
import pickle
with open('../models/train_test_data.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train_balanced,
        'X_test': X_test,
        'y_train': y_train_balanced,
        'y_test': y_test,
        'feature_names': X.columns.tolist()
    }, f)

print("Saved: models/train_test_data.pkl")

print("\nData Processing Done")
print("-"*50)
print(f"Summary:")
print(f"Original data: {len(df_encoded)} samples")
print(f"Training data: {len(X_train_balanced)} samples (after SMOTE)")
print(f"Testing data: {len(X_test)} samples")
print(f"Features: {len(X.columns)}")

Saving Processed Data
----------------------------------------------------------------------
Saved: data/processed/telco_churn_cleaned.csv
Saved: models/train_test_data.pkl

Data Processing Done
--------------------------------------------------
Summary:
Original data: 7043 samples
Training data: 8278 samples (after SMOTE)
Testing data: 1409 samples
Features: 19
